# Notebook 01: UCI Human Activity Recognition Dataset Exploration

**Project:** Human Activity Recognition from Wearable IMU Sensor Data using Time-Series Machine Learning  
**Dataset:** UCI HAR Using Smartphones (DOI: 10.24432/C54S4K)  
**Objective:** Verify data loading, inspect tensor dimensions, validate subject disjointness, explore sensor channel waveforms, and examine class distributions.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.load_dataset import (
    load_activity_labels,
    load_feature_names,
    load_full_split,
    resolve_dataset_dir,
    CHANNEL_NAMES,
)
from src.data.validate_dataset import validate_splits

sns.set_theme(style="whitegrid")
print(f"Project root: {project_root}")

## 1. Load Activity Labels and Feature Names

In [ ]:
dataset_dir = resolve_dataset_dir()
activity_map = load_activity_labels(dataset_dir)
feature_names = load_feature_names(dataset_dir)

print("Activity Mapping:")
for k, v in activity_map.items():
    print(f"  {k}: {v}")

print(f"\nTotal Precomputed Features: {len(feature_names)}")
print(f"First 5 features: {feature_names[:5]}")

## 2. Load Inertial Signal Tensors, Labels, and Subjects

In [ ]:
X_train, y_train, subj_train, _ = load_full_split(dataset_dir, split="train", load_features=False)
X_test, y_test, subj_test, _ = load_full_split(dataset_dir, split="test", load_features=False)

print(f"X_train shape: {X_train.shape} (Expected: 7352, 128, 9)")
print(f"y_train shape: {y_train.shape}")
print(f"subj_train shape: {subj_train.shape}")
print(f"X_test shape:  {X_test.shape} (Expected: 2947, 128, 9)")
print(f"y_test shape:  {y_test.shape}")
print(f"subj_test shape: {subj_test.shape}")

## 3. Dataset Validation & Subject Disjointness Check

In [ ]:
report = validate_splits(X_train, y_train, subj_train, X_test, y_test, subj_test)
print(report.summary())

## 4. Activity Class Distribution

In [ ]:
act_labels = [activity_map[i] for i in range(1, 7)]
train_counts = [(y_train == i).sum() for i in range(1, 7)]
test_counts = [(y_test == i).sum() for i in range(1, 7)]

df_dist = pd.DataFrame({
    "Activity": act_labels,
    "Train": train_counts,
    "Test": test_counts,
    "Total": [a + b for a, b in zip(train_counts, test_counts)]
})
display(df_dist)

## 5. Waveform Visualization: Stationary vs. Dynamic Motion

In [ ]:
time_axis = np.arange(128) / 50.0  # 50 Hz sampling -> 2.56 s duration
walk_idx = np.where(y_train == 1)[0][0]  # WALKING
sit_idx = np.where(y_train == 4)[0][0]   # SITTING

fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharey="row")

# Total Acceleration
axes[0, 0].plot(time_axis, X_train[sit_idx, :, 6:9], label=["Acc X", "Acc Y", "Acc Z"])
axes[0, 0].set_title("Stationary: SITTING - Total Acceleration (g)", fontweight="bold")
axes[0, 0].set_ylabel("Acc (g)")
axes[0, 0].legend()

axes[0, 1].plot(time_axis, X_train[walk_idx, :, 6:9], label=["Acc X", "Acc Y", "Acc Z"])
axes[0, 1].set_title("Dynamic: WALKING - Total Acceleration (g)", fontweight="bold")
axes[0, 1].legend()

# Gyroscope
axes[1, 0].plot(time_axis, X_train[sit_idx, :, 3:6], label=["Gyro X", "Gyro Y", "Gyro Z"])
axes[1, 0].set_title("Stationary: SITTING - Gyroscope (rad/s)", fontweight="bold")
axes[1, 0].set_ylabel("Angular Velocity (rad/s)")
axes[1, 0].set_xlabel("Time (seconds)")
axes[1, 0].legend()

axes[1, 1].plot(time_axis, X_train[walk_idx, :, 3:6], label=["Gyro X", "Gyro Y", "Gyro Z"])
axes[1, 1].set_title("Dynamic: WALKING - Gyroscope (rad/s)", fontweight="bold")
axes[1, 1].set_xlabel("Time (seconds)")
axes[1, 1].legend()

plt.tight_layout()
plt.show()